## Imports og tilkobling

In [2]:
import sys
import os
import pandas as pd
import io
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

sys.path.append(os.path.abspath("../../../"))
from src.feature_engineering.forbruksdata import create_consumption_features

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_name = "trafoforbruksdata"

## Hente CSV fra blob

In [4]:
blob_name = "TIMENES TS.csv"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=blob_name
)

data = blob_client.download_blob().readall()

df = pd.read_csv(io.BytesIO(data))

## Feature engineering

In [5]:
# feature engineering
df = create_consumption_features(df)

# se resultat
df.head()

,MeteringPointAnonymous,TimestampUtc,Value,TransformerStation,ConsumptionCode,hour,weekday,month,is_weekend,is_holiday,day_night
0,e7b9ef5e-4808-4011-bf81-7e3058b62a3e,2025-12-25 23:00:00,1.27,TIMENES TS,35,23,3,12,False,False,night
1,e7b9ef5e-4808-4011-bf81-7e3058b62a3e,2025-12-26 00:00:00,2.16,TIMENES TS,35,0,4,12,False,False,night
2,e7b9ef5e-4808-4011-bf81-7e3058b62a3e,2025-12-26 01:00:00,1.63,TIMENES TS,35,1,4,12,False,False,night
3,e7b9ef5e-4808-4011-bf81-7e3058b62a3e,2025-12-26 02:00:00,1.90,TIMENES TS,35,2,4,12,False,False,night
4,e7b9ef5e-4808-4011-bf81-7e3058b62a3e,2025-12-26 03:00:00,1.98,TIMENES TS,35,3,4,12,False,False,night


## Legge til blob

In [6]:
import io

# --- Lag CSV i minnet ---
csv_buffer = io.BytesIO()
df.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)  # gå til starten av bufferet

# --- Last opp til blob ---
output_blob_name = "TIMENES_processed.csv"  # kan bruke samme navn eller nytt
blob_client = blob_service_client.get_blob_client(container=container_name, blob=output_blob_name)
blob_client.upload_blob(csv_buffer, overwrite=True)  # overwrite=True overskriver eksisterende fil

print(f"{output_blob_name} lastet opp til blob!")

TIMENES_processed.csv lastet opp til blob!
